In [38]:
import pandas as pd
from sqlalchemy import create_engine

In [39]:
# Loading background variables from my hidden .env file
import os

from dotenv import load_dotenv


load_dotenv()

db_user = os.getenv("db_user")
db_password = os.getenv("db_password")
db_host = os.getenv("db_host")
db_port = os.getenv("db_port")
db_name = os.getenv("db_name")

In [40]:
# Building the connection string securely
connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(connection_string)


In [41]:
# Loading the local CSV from my Jupyter notebook 
#csv_file_path = "fleet_telematics_database.csv"
df = pd.read_csv('fleet_telematics_database.csv')

In [42]:
# top five rows of the dataframe
df.head()

,Timestamp,Truck_ID,Fleet_Status,Origin_Depot,Assigned_Route,Total_Route_Distance_KM,Distance_Covered_KM,Current_Speed_KMH,Fuel_Volume_Lost_Liters,Fuel_Theft_Anomaly,Engine_Temp_C,Hours_Since_Service,Maintenance_Recommendation,Vibration_G,Breakdown_Risk_Target,Actual_Trip_Duration_Hours
0,2026-01-01 00:00:00,OG-FLEET-1000,Off-Duty,Warri Delta Depot,Sagamu-Benin Express Corridor,260,0.0,0,0.3,0,47.5,129,Optimal Operations,0.19,0,0.0
1,2026-01-01 04:00:00,OG-FLEET-1000,Off-Duty,Warri Delta Depot,Sagamu-Benin Express Corridor,260,0.0,0,1.0,0,48.2,129,Optimal Operations,0.35,0,0.0
2,2026-01-01 08:00:00,OG-FLEET-1000,Off-Duty,Warri Delta Depot,Sagamu-Benin Express Corridor,260,0.0,0,1.5,0,43.8,129,Optimal Operations,0.18,0,0.0
3,2026-01-01 12:00:00,OG-FLEET-1000,Off-Duty,Warri Delta Depot,Sagamu-Benin Express Corridor,260,0.0,0,0.5,0,52.9,129,Optimal Operations,0.28,0,0.0
4,2026-01-01 16:00:00,OG-FLEET-1000,Off-Duty,Warri Delta Depot,Sagamu-Benin Express Corridor,260,0.0,0,1.1,0,42.7,129,Optimal Operations,0.25,0,0.0


In [43]:
df.columns

Index(['Timestamp', 'Truck_ID', 'Fleet_Status', 'Origin_Depot',
       'Assigned_Route', 'Total_Route_Distance_KM', 'Distance_Covered_KM',
       'Current_Speed_KMH', 'Fuel_Volume_Lost_Liters', 'Fuel_Theft_Anomaly',
       'Engine_Temp_C', 'Hours_Since_Service', 'Maintenance_Recommendation',
       'Vibration_G', 'Breakdown_Risk_Target', 'Actual_Trip_Duration_Hours'],
      dtype='object')

In [44]:
df.columns = [col.lower() for col in df.columns]
df.columns

Index(['timestamp', 'truck_id', 'fleet_status', 'origin_depot',
       'assigned_route', 'total_route_distance_km', 'distance_covered_km',
       'current_speed_kmh', 'fuel_volume_lost_liters', 'fuel_theft_anomaly',
       'engine_temp_c', 'hours_since_service', 'maintenance_recommendation',
       'vibration_g', 'breakdown_risk_target', 'actual_trip_duration_hours'],
      dtype='object')

In [45]:
# Streaming securely to PostgreSQL
try:
    print("Initiating secure data stream to PostgreSQL database")
    df.to_sql(
        name='fleet_telematics_logs', 
        con=engine, 
        if_exists='append', 
        index=False,
        schema='public'
    )
    print("ETL Migration Complete! Passwords remained protected.")
except Exception as e:
    print(f"Connection Failed: Ensure your .env variables match your local configurations. Details: {e}")

Initiating secure data stream to PostgreSQL database
ETL Migration Complete! Passwords remained protected.


In [46]:
#!pip install plotly pandas openpyxl


In [47]:
import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
# Save Compiled Models for Web App Use
import joblib 


In [48]:
#%pip install nbformat



Step 3: Test SQL Pull & Prototype Plotly Express Charts

Before coding a web app, you must verify that your data can be pulled back out of PostgreSQL and that your Plotly figures render perfectly. This cell tests your database queries and creates interactive dashboard figures right inside your notebook.

In [49]:
# Pulling data live from the database to ensure connection integrity
df_db = pd.read_sql("SELECT * FROM public.fleet_telematics_logs;", con=engine)

# Prototyping Chart 1: Macro Fleet Status Distribution Pie Chart
status_summary = df_db.groupby("fleet_status")["truck_id"].nunique().reset_index()
fig_pie = px.pie(
    status_summary, 
    names="fleet_status", 
    values="truck_id", 
    hole=0.45,
    title="Fleet Status Distribution Operational Profile",
    color_discrete_sequence=px.colors.sequential.Bluyl_r
)
fig_pie.show() # Renders chart interactively inside the notebook cell

# Prototyping Chart 2: Total Refined Fuel Loss/Theft Bar Chart
theft_by_depot = df_db[df_db["fuel_theft_anomaly"] == 1].groupby("origin_depot")["fuel_volume_lost_liters"].sum().reset_index()
fig_bar = px.bar(
    theft_by_depot, 
    x="origin_depot", 
    y="fuel_volume_lost_liters", 
    color="origin_depot",
    title="Volumetric Product Theft Losses across Refining/Depot Yards",
    labels={"fuel_volume_lost_liters": "Total Refined Liters Siphoned"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig_bar.show()
print("✅ Database query successful. Interactive Plotly charts validated and working.")


✅ Database query successful. Interactive Plotly charts validated and working.



Step 4: Train, Evaluate, and Serialize Machine Learning Models

This cell splits your data, handles one-hot encoding for categorical routing vectors, trains your predictive models, prints diagnostic metrics to ensure accuracy, and exports the final .pkl model artifacts

In [50]:
df_db["breakdown_risk_target"].value_counts()

breakdown_risk_target
0    53589
1     4911
Name: count, dtype: int64

In [51]:
df.columns

Index(['timestamp', 'truck_id', 'fleet_status', 'origin_depot',
       'assigned_route', 'total_route_distance_km', 'distance_covered_km',
       'current_speed_kmh', 'fuel_volume_lost_liters', 'fuel_theft_anomaly',
       'engine_temp_c', 'hours_since_service', 'maintenance_recommendation',
       'vibration_g', 'breakdown_risk_target', 'actual_trip_duration_hours'],
      dtype='object')

In [ ]:
# ==========================================
# MODEL A: PREDICTIVE MAINTENANCE CLASSIFIER (UPDATED)
# ==========================================
# Filter for trucks actively moving to prevent status leakage
transit_only = df_db[df_db["fleet_status"].str.contains("In Transit", na=False)].copy()

# Included distance_covered_km based on your brilliant structural insight!
X_class = transit_only[[
    "current_speed_kmh", 
    "engine_temp_c", 
    "hours_since_service", 
    # "vibration_g", 
    "total_route_distance_km",
    "distance_covered_km"
]]
y_class = transit_only["breakdown_risk_target"]

# Stratify ensure balanced train/test splits for our minority breakdown class
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

clf = XGBClassifier(n_estimators=80, max_depth=4, random_state=42, learning_rate=0.1)
clf.fit(X_train_c, y_train_c)

print("--- 1. Predictive Maintenance Classifier Performance ---")
print(classification_report(y_test_c, clf.predict(X_test_c)))


--- 1. Predictive Maintenance Classifier Performance ---
              precision    recall  f1-score   support

           0       0.99      0.91      0.95      6749
           1       0.61      0.95      0.75       982

    accuracy                           0.92      7731
   macro avg       0.80      0.93      0.85      7731
weighted avg       0.94      0.92      0.92      7731



In [53]:
from sklearn.metrics import accuracy_score

# Calculating training accuracy
train_acc = accuracy_score(y_train_c, clf.predict(X_train_c))
print(f"📊 Your Exact Training Accuracy is: {round(train_acc * 100, 2)}%")


📊 Your Exact Training Accuracy is: 92.14%


In [54]:
# Check feature importance to see which column holds 100% of the weight
importances = pd.Series(clf.feature_importances_, index=X_class.columns)
print("--- Model A Feature Importance ---")
print(importances.sort_values(ascending=False))


--- Model A Feature Importance ---
hours_since_service        0.877994
current_speed_kmh          0.056238
engine_temp_c              0.038204
distance_covered_km        0.013925
total_route_distance_km    0.013639
dtype: float32


In [55]:
# MODEL B: DYNAMIC TRANSIT TIME REGRESSOR
# ==========================================
# One-hot encode routes for ETA calculations
transit_df = transit_only.copy()
transit_df = pd.get_dummies(transit_df, columns=['assigned_route'], drop_first=True)
route_features = [col for col in transit_df.columns if 'assigned_route_' in col]

X_reg = transit_df[["total_route_distance_km", "current_speed_kmh"] + route_features]
y_reg = transit_df["actual_trip_duration_hours"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=50, random_state=42)
reg.fit(X_train_r, y_train_r)

mae = mean_absolute_error(y_test_r, reg.predict(X_test_r))
print(f"--- 2. Regressor Performance ---")
print(f"Mean Absolute Error (MAE): {round(mae, 2)} Hours (~{round(mae * 60)} mins variance)\n")

--- 2. Regressor Performance ---
Mean Absolute Error (MAE): 1.0 Hours (~60 mins variance)



In [56]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# 1. Generate predictions for both splits to check variance visually
y_train_pred_r = reg.predict(X_train_r)
y_test_pred_r = reg.predict(X_test_r)

# 2. Compile Test Results into a DataFrame for Plotly
df_results = pd.DataFrame({
    "Actual_Duration": y_test_r,
    "Predicted_Duration": y_test_pred_r,
    "Residual_Error": y_test_r - y_test_pred_r
})

# CHART 1: Actual vs. Predicted Transit Times
fig_regression = px.scatter(
    df_results, 
    x="Actual_Duration", 
    y="Predicted_Duration",
    labels={"Actual_Duration": "Actual Trip Duration (Hours)", "Predicted_Duration": "Model Predicted Duration (Hours)"},
    title="Model B Evaluation: Actual vs. Predicted Transit Times",
    opacity=0.6,
    color_discrete_sequence=["#1f77b4"]
)

# Add a perfect diagonal reference line (Where Actual == Predicted)
min_val = min(df_results["Actual_Duration"].min(), df_results["Predicted_Duration"].min())
max_val = max(df_results["Actual_Duration"].max(), df_results["Predicted_Duration"].max())
fig_regression.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode="lines", name="Perfect Forecast Line", line=dict(color="red", dash="dash"))
)
fig_regression.show()

# CHART 2: Residual Error Distribution (Checks for systematic bias)
fig_residuals = px.histogram(
    df_results, 
    x="Residual_Error",
    nbins=30,
    title="Distribution of ETA Residual Errors (Target: Zero-Centered Gaussian)",
    labels={"Residual_Error": "Prediction Error (Actual - Predicted Hours)"},
    color_discrete_sequence=["#2ca02c"]
)
fig_residuals.show()

print("✅ Model B evaluation charts successfully rendered.")


✅ Model B evaluation charts successfully rendered.


In [57]:
theft_df = pd.get_dummies(df_db, columns=['assigned_route', 'fleet_status'], drop_first=True)


In [58]:
# ==========================================
# MODEL C: FUEL THEFT ANOMALY PREDICTOR (NEW!)
# ==========================================
# We include fleet_status here since theft often occurs when 'Off-Duty' or 'Broken Down'
theft_df = pd.get_dummies(df_db, columns=['assigned_route', 'fleet_status'], drop_first=True)
theft_route_cols = [col for col in theft_df.columns if 'assigned_route_' in col]
theft_status_cols = [col for col in theft_df.columns if 'fleet_status_' in col]

# Features look at vehicle state context, EXCLUDING the exact siphoned volume to avoid leakage
X_theft = theft_df[["current_speed_kmh", "engine_temp_c"] + theft_route_cols + theft_status_cols]
y_theft = theft_df["fuel_theft_anomaly"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_theft, y_theft, test_size=0.2, random_state=42, stratify=y_theft)

theft_clf = XGBClassifier(n_estimators=80, max_depth=4, random_state=42, learning_rate=0.1)
theft_clf.fit(X_train_t, y_train_t)

print("--- 3. Fuel Theft Anomaly Predictor Performance ---")
print(classification_report(y_test_t, theft_clf.predict(X_test_t)))

--- 3. Fuel Theft Anomaly Predictor Performance ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     11180
           1       1.00      0.72      0.84       520

    accuracy                           0.99     11700
   macro avg       0.99      0.86      0.92     11700
weighted avg       0.99      0.99      0.99     11700



In [59]:
# EXPORT ALL THREE COMPILED MODELS
# ==========================================
joblib.dump(clf, "maintenance_model.pkl")
joblib.dump(reg, "eta_model.pkl")
joblib.dump(theft_clf, "fuel_theft_model.pkl")

print("✅ All 3 Machine Learning models successfully verified, exported, and saved.")

✅ All 3 Machine Learning models successfully verified, exported, and saved.



Step 5: Test Live Inference Logic (Mock App Inputs)

Before pasting your prediction buttons into Streamlit, test your raw inference functions directly inside the notebook with mock inputs to guarantee they output the correct risk scores and hour durations

In [ ]:
import numpy as np
import pandas as pd

# =====================================================================
# 1. TEST MOCK INPUTS FOR MECHANICAL BREAKDOWN CLASSIFIER (MODEL A)
# =====================================================================
# Features must match: ["current_speed_kmh", "engine_temp_c", "hours_since_service", "total_route_distance_km", "distance_covered_km"]
# Mock: Moving at 65km/h, running hot at 115°C, overdue for check at 380 hours, 130km total route, has driven 85km
mock_telemetry_input = np.array([[65, 115, 380, 130.0, 85.0]]) 

risk_prob = clf.predict_proba(mock_telemetry_input)[0][1]
print(f"🧪 Model A (Breakdown) Test Prediction Risk: {round(risk_prob * 100, 2)}%")


# =====================================================================
# 2. TEST MOCK INPUTS FOR TRAVEL DURATION REGRESSOR (MODEL B)
# =====================================================================
# Features must match: ["total_route_distance_km", "current_speed_kmh", "assigned_route_Port Harcourt-Aba Road Corridor", "assigned_route_Sagamu-Benin Express Corridor"]
# Mock: Lagos-Ibadan Express Corridor (130km distance, so both route column flags remain 0) moving at 65km/h
mock_route_input = np.array([[130.0, 65, 0, 0]]) 

predicted_duration = reg.predict(mock_route_input)[0]
print(f"🧪 Model B (ETA) Test Predicted Duration: {round(predicted_duration, 2)} Hours")


# =====================================================================
# 3. TEST MOCK INPUTS FOR FUEL THEFT ANOMALY PREDICTOR (MODEL C)
# =====================================================================
# Let's inspect the exact column names required for Model C to ensure the array is perfect
theft_features = X_theft.columns.tolist()

# Mock: Truck is stopped (speed=0), normal engine temperature (45°C)
# We will create a single row DataFrame to safely map the columns without feature shape mismatch errors
mock_theft_df = pd.DataFrame(0, index=[0], columns=theft_features)
mock_theft_df["current_speed_kmh"] = 0
mock_theft_df["engine_temp_c"] = 45.0
# Simulate vehicle being stopped mid-transit by toggling the status column flag to 1
if "fleet_status_In Transit (Stopped/Unscheduled)" in mock_theft_df.columns:
    mock_theft_df["fleet_status_In Transit (Stopped/Unscheduled)"] = 1

theft_risk_prob = theft_clf.predict_proba(mock_theft_df)[0][1]
print(f"🧪 Model C (Fuel Theft) Test Prediction Risk: {round(theft_risk_prob * 100, 2)}%")

print("\n✅ All notebook validation tasks passed. Code blocks successfully verified!")


🧪 Model A (Breakdown) Test Prediction Risk: 69.41000366210938%
🧪 Model B (ETA) Test Predicted Duration: 3.91 Hours
🧪 Model C (Fuel Theft) Test Prediction Risk: 99.93000030517578%

✅ All notebook validation tasks passed. Code blocks successfully verified!


c:\Users\DONATUS\anaconda24\envs\mlops_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


: 